In [1]:
from primaite.agents.aegis.gllm import GLLM
from primaite.agents.aegis.modules.openai import OpenAIClient
from primaite.agents.git_agent import GITAgent
from torch.utils.data import DataLoader
from primaite.agents.llm.utils import network_connectivity_desc
import logging

logging.disable(logging.CRITICAL)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-08-23 14:00:46.582206: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-23 14:00:46.629418: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-23 14:00:47.332161: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has 

In [2]:
gllm = GLLM()
openai = OpenAIClient(openai_api_key="")

In [3]:
questions = [
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "Given the provided network configuration, how many nodes are their in this network?"
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
    "How many nodes are in the network?",
]

In [4]:
# Mock a primaite graph for development
agent = GITAgent(
    training_config_path="../src/primaite/config/_package_data/training/git.yaml",
    lay_down_config_path="../src/primaite/config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml",
)
obs = agent._env.reset()
data = agent.create_graph(obs)
network_desc = network_connectivity_desc(agent._env)

<Figure size 640x480 with 0 Axes>

In [5]:
import os
import pickle as pkl

if "openai_responses.pkl" not in os.listdir("./"):
    openai_responses = []
    openai_prompts = gllm.build_prompts(questions=questions, network_desc=network_desc, model="openai")
    for prompt in openai_prompts:
        openai_responses.append(openai.generate(prompt=prompt))

    with open("openai_responses.pkl", "wb") as file:
        pkl.dump(openai_responses, file)
else:
    with open("openai_responses.pkl", "rb") as file:
        openai_responses = pkl.load(file)

In [6]:
from primaite.agents.aegis.data import GLLMDataset, collate_fn

dataset = GLLMDataset(
    graphs=[data for _ in range(len(questions))],
    questions=[question for question in questions],
    gt_answers=[response for response in openai_responses],
    llm=gllm.llm,
)

In [7]:
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

In [8]:
from primaite.agents.aegis.gllm import train_loop

gllm_responses = train_loop(model=gllm, dataloader=dataloader, network_desc=network_desc, n_epochs=10)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
